In [2]:
!pip install fastf1 pandas numpy scikit-learn matplotlib seaborn joblib --quiet

In [3]:
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import joblib, os, warnings
warnings.filterwarnings('ignore')

fastf1.Cache.enable_cache('f1_cache')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)

print("All good!")

All good!


In [4]:
# Load clean laps from module 01
df = pd.read_csv('data/clean_laps.csv')

# Load Kaggle CSVs for historical pit stop data
pit_stops    = pd.read_csv('data/pit_stops.csv')
results      = pd.read_csv('data/results.csv')
races        = pd.read_csv('data/races.csv')
drivers      = pd.read_csv('data/drivers.csv')
constructors = pd.read_csv('data/constructors.csv')

print(f"Clean laps (FastF1):  {df.shape[0]}")
print(f"Pit stops (Kaggle):   {pit_stops.shape[0]}")
print(f"Results (Kaggle):     {results.shape[0]}")
print(pit_stops.head())

Clean laps (FastF1):  5495
Pit stops (Kaggle):   11371
Results (Kaggle):     26759
   raceId  driverId  stop  lap      time duration  milliseconds
0     841       153     1    1  17:05:23   26.898         26898
1     841        30     1    1  17:05:52   25.021         25021
2     841        17     1   11  17:20:48   23.426         23426
3     841         4     1   12  17:22:34   23.251         23251
4     841        13     1   13  17:24:10   23.842         23842


In [5]:
# Races to pull from FastF1 — recent seasons
fastf1_races = [
    # 2022
    (2022, 'Bahrain'), (2022, 'Saudi Arabia'), (2022, 'Australia'),
    (2022, 'Spain'), (2022, 'Monaco'), (2022, 'Britain'),
    (2022, 'Hungary'), (2022, 'Belgium'), (2022, 'Italy'),
    # 2023
    (2023, 'Bahrain'), (2023, 'Saudi Arabia'), (2023, 'Australia'),
    (2023, 'Spain'), (2023, 'Monaco'), (2023, 'Britain'),
    (2023, 'Hungary'), (2023, 'Italy'), (2023, 'Singapore'),
    # 2024
    (2024, 'Bahrain'), (2024, 'Saudi Arabia'), (2024, 'Australia'),
    (2024, 'Spain'), (2024, 'Monaco'), (2024, 'Britain'),
    (2024, 'Hungary'), (2024, 'Italy'), (2024, 'Singapore'),
    # 2026
    (2026, 'Australia'), (2026, 'China'), (2026, 'Japan'),
]

all_sessions = []

for year, race in fastf1_races:
    print(f"Loading {race} {year}...")
    try:
        session = fastf1.get_session(year, race, 'R')
        session.load(telemetry=False, weather=False, messages=False)
        laps = session.laps.copy()
        laps['race_name'] = f"{race}{year}"
        laps['year'] = year
        all_sessions.append(laps)
        print(f"  Done — {len(laps)} laps")
    except Exception as e:
        print(f"  Skipped {race} {year}: {e}")

df_recent = pd.concat(all_sessions, ignore_index=True)
print(f"\nTotal recent laps: {df_recent.shape[0]}")
print(f"Years covered: {sorted(df_recent['year'].unique())}")

Loading Bahrain 2022...


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

  Done — 1125 laps
Loading Saudi Arabia 2022...


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

  Done — 820 laps
Loading Australia 2022...


...

  Done — 1007 laps
Loading China 2026...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 12 completed the race distance 00:00.022000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '87', '10', '30', '6', '55', '43', '27', '41', '77', '31', '11', '3', '14', '18', '81', '1', '5', '23']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


  Done — 924 laps
Loading Japan 2026...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 22 drivers: ['12', '81', '16', '63', '1', '44', '10', '3', '30', '31', '27', '6', '5', '41', '55', '43', '11', '14', '77', '23', '18', '87']


  Done — 1107 laps

Total recent laps: 32572
Years covered: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2026)]


In [6]:
def extract_strategy_features(df):
    """
    For each driver in each race, build a row per lap describing
    the full strategic situation — used to predict optimal pit window
    """
    records = []
    
    for race in df['race_name'].unique():
        race_df = df[df['race_name'] == race].copy()
        race_df = race_df.dropna(subset=['LapTime', 'TyreLife', 'Compound', 'Position'])
        race_df['LapTimeSec'] = pd.to_timedelta(race_df['LapTime']).dt.total_seconds()
        race_df['Position'] = pd.to_numeric(race_df['Position'], errors='coerce')
        race_df = race_df.dropna(subset=['LapTimeSec', 'Position'])
        
        total_laps = race_df['LapNumber'].max()
        
        for driver in race_df['Driver'].unique():
            driver_df = race_df[race_df['Driver'] == driver].copy()
            driver_df = driver_df.sort_values('LapNumber')
            
            # Find pit laps — where TyreLife resets
            driver_df['is_pit_lap'] = (
                driver_df['TyreLife'].diff() < 0
            ).astype(int)
            
            for _, row in driver_df.iterrows():
                lap = row['LapNumber']
                
                # Race progress (0 to 1)
                race_progress = lap / total_laps if total_laps > 0 else 0
                
                # Laps remaining
                laps_remaining = total_laps - lap
                
                # Tire age normalized
                tire_age = row['TyreLife']
                tire_age_normalized = tire_age / 50.0
                
                # Compound
                compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 
                               'INTERMEDIATE': 3, 'WET': 4}
                compound = compound_map.get(row['Compound'], 1)
                
                # Position normalized
                position = row['Position'] / 20.0
                
                # Lap time delta from driver's race median
                median_time = driver_df['LapTimeSec'].median()
                lap_delta = row['LapTimeSec'] - median_time
                
                # Is this a pit lap? (label)
                label = row['is_pit_lap']
                
                records.append({
                    'race': race,
                    'driver': driver,
                    'lap': lap,
                    'race_progress': round(race_progress, 3),
                    'laps_remaining': laps_remaining,
                    'tire_age': tire_age,
                    'tire_age_normalized': round(tire_age_normalized, 3),
                    'compound': compound,
                    'position_normalized': round(position, 3),
                    'lap_delta': round(lap_delta, 3),
                    'should_pit': label
                })
    
    return pd.DataFrame(records)

print("Building strategy dataset...")
strategy_df = extract_strategy_features(df_recent)
print(f"Total rows: {strategy_df.shape[0]}")
print(f"Pit stop rate: {strategy_df['should_pit'].mean():.1%}")
print(strategy_df.head(10))

Building strategy dataset...
Total rows: 31976
Pit stop rate: 3.0%
          race driver   lap  race_progress  laps_remaining  tire_age  \
0  Bahrain2022    VER   1.0          0.018            56.0       4.0   
1  Bahrain2022    VER   2.0          0.035            55.0       5.0   
2  Bahrain2022    VER   3.0          0.053            54.0       6.0   
3  Bahrain2022    VER   4.0          0.070            53.0       7.0   
4  Bahrain2022    VER   5.0          0.088            52.0       8.0   
5  Bahrain2022    VER   6.0          0.105            51.0       9.0   
6  Bahrain2022    VER   7.0          0.123            50.0      10.0   
7  Bahrain2022    VER   8.0          0.140            49.0      11.0   
8  Bahrain2022    VER   9.0          0.158            48.0      12.0   
9  Bahrain2022    VER  10.0          0.175            47.0      13.0   

   tire_age_normalized  compound  position_normalized  lap_delta  should_pit  
0                 0.08         0                  0.1      1.

In [7]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

FEATURES = [
    'race_progress', 'laps_remaining', 'tire_age',
    'tire_age_normalized', 'compound',
    'position_normalized', 'lap_delta'
]
TARGET = 'should_pit'

X = strategy_df[FEATURES]
y = strategy_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\nFeature importances:")
print(fi.round(3))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      6207
           1       0.77      0.96      0.85       189

    accuracy                           0.99      6396
   macro avg       0.88      0.98      0.92      6396
weighted avg       0.99      0.99      0.99      6396

ROC-AUC: 0.999

Feature importances:
lap_delta              0.725
tire_age               0.121
tire_age_normalized    0.072
race_progress          0.060
position_normalized    0.010
laps_remaining         0.007
compound               0.006
dtype: float64


In [8]:
FEATURES = [
    'race_progress', 'laps_remaining', 'tire_age',
    'tire_age_normalized', 'compound', 'position_normalized'
]
TARGET = 'should_pit'

X = strategy_df[FEATURES]
y = strategy_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\nFeature importances:")
print(fi.round(3))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      6207
           1       0.46      0.92      0.61       189

    accuracy                           0.97      6396
   macro avg       0.73      0.94      0.80      6396
weighted avg       0.98      0.97      0.97      6396

ROC-AUC: 0.991

Feature importances:
tire_age_normalized    0.431
tire_age               0.378
race_progress          0.134
position_normalized    0.024
compound               0.018
laps_remaining         0.015
dtype: float64


In [10]:
print(strategy_df.columns.tolist())
print(df_recent[['Driver', 'Team', 'race_name']].dropna().drop_duplicates().head(10))
print(df_recent['Team'].unique())

['race', 'driver', 'lap', 'race_progress', 'laps_remaining', 'tire_age', 'tire_age_normalized', 'compound', 'position_normalized', 'lap_delta', 'should_pit']
    Driver             Team    race_name
0      VER  Red Bull Racing  Bahrain2022
54     GAS       AlphaTauri  Bahrain2022
99     PER  Red Bull Racing  Bahrain2022
156    ALO           Alpine  Bahrain2022
213    LEC          Ferrari  Bahrain2022
270    STR     Aston Martin  Bahrain2022
327    MAG     Haas F1 Team  Bahrain2022
384    TSU       AlphaTauri  Bahrain2022
441    ALB         Williams  Bahrain2022
498    ZHO       Alfa Romeo  Bahrain2022
['Red Bull Racing' 'AlphaTauri' 'Alpine' 'Ferrari' 'Aston Martin'
 'Haas F1 Team' 'Williams' 'Alfa Romeo' 'McLaren' 'Mercedes' 'Kick Sauber'
 'RB' 'Cadillac' 'Audi' 'Racing Bulls']


In [11]:
# Get driver-team mapping per race
team_info = df_recent[['Driver', 'Team', 'race_name']].dropna().drop_duplicates()

# Merge into strategy_df
strategy_df = strategy_df.merge(
    team_info,
    left_on=['driver', 'race'],
    right_on=['Driver', 'race_name'],
    how='left'
).drop(columns=['Driver', 'race_name'])

# Encode team
team_map = {
    'Red Bull Racing': 0, 'Ferrari': 1, 'Mercedes': 2,
    'McLaren': 3, 'Aston Martin': 4, 'Alpine': 5,
    'Williams': 6, 'AlphaTauri': 7, 'Alfa Romeo': 8,
    'Haas F1 Team': 9, 'RB': 7, 'Kick Sauber': 8,
    'Racing Bulls': 7, 'Cadillac': 10, 'Audi': 11
}
strategy_df['team_num'] = strategy_df['Team'].map(team_map).fillna(5).astype(int)

print(f"Team merge complete: {strategy_df.shape[0]} rows")
print(strategy_df[['driver', 'Team', 'team_num']].drop_duplicates().head(12))

Team merge complete: 31976 rows
    driver             Team  team_num
0      VER  Red Bull Racing         0
53     GAS       AlphaTauri         7
97     PER  Red Bull Racing         0
153    ALO           Alpine         5
210    LEC          Ferrari         1
265    STR     Aston Martin         4
322    MAG     Haas F1 Team         9
379    TSU       AlphaTauri         7
436    ALB         Williams         6
493    ZHO       Alfa Romeo         8
549    HUL     Aston Martin         4
606    RIC          McLaren         3


In [12]:
FEATURES = [
    'race_progress', 'laps_remaining',
    'tire_age', 'compound',
    'position_normalized', 'team_num'
]
TARGET = 'should_pit'

X = strategy_df[FEATURES]
y = strategy_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_weights)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\nFeature importances:")
print(fi.round(3))

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.97      0.98      6207
           1       0.48      0.91      0.63       189

    accuracy                           0.97      6396
   macro avg       0.74      0.94      0.80      6396
weighted avg       0.98      0.97      0.97      6396

ROC-AUC: 0.991

Feature importances:
tire_age               0.806
race_progress          0.133
position_normalized    0.022
compound               0.017
laps_remaining         0.014
team_num               0.008
dtype: float64


In [13]:
def build_optimal_pit_dataset(df):
    """
    For every actual pit stop, determine if it was OPTIMAL.
    Optimal = driver maintained or gained position within 5 laps after pit.
    Suboptimal = driver lost 2+ positions within 5 laps after pit.
    """
    records = []

    for race in df['race_name'].unique():
        race_df = df[df['race_name'] == race].copy()
        race_df = race_df.dropna(subset=['Position', 'TyreLife'])
        race_df['Position'] = pd.to_numeric(race_df['Position'], errors='coerce')
        race_df = race_df.dropna(subset=['Position'])
        race_df['Position'] = race_df['Position'].astype(int)
        
        total_laps = race_df['LapNumber'].max()

        for driver in race_df['Driver'].unique():
            driver_df = race_df[race_df['Driver'] == driver].copy()
            driver_df = driver_df.sort_values('LapNumber')
            
            # Find actual pit laps
            driver_df['tyre_reset'] = driver_df['TyreLife'].diff() < 0
            pit_laps = driver_df[driver_df['tyre_reset']]['LapNumber'].values

            for pit_lap in pit_laps:
                # Get situation BEFORE pit
                before = driver_df[driver_df['LapNumber'] == pit_lap - 1]
                after_window = driver_df[
                    (driver_df['LapNumber'] >= pit_lap + 1) &
                    (driver_df['LapNumber'] <= pit_lap + 5)
                ]

                if before.empty or after_window.empty:
                    continue

                pos_before = before.iloc[0]['Position']
                pos_after = after_window.iloc[-1]['Position']
                tire_age_at_pit = before.iloc[0]['TyreLife']
                compound_at_pit = before.iloc[0]['Compound']
                race_progress = pit_lap / total_laps if total_laps > 0 else 0

                compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2,
                               'INTERMEDIATE': 3, 'WET': 4}
                compound_num = compound_map.get(compound_at_pit, 1)

                # Label: 1 = optimal (gained or held position), 0 = suboptimal
                position_delta = pos_before - pos_after  # positive = gained positions
                optimal = 1 if position_delta >= 0 else 0

                records.append({
                    'race': race,
                    'driver': driver,
                    'pit_lap': pit_lap,
                    'tire_age_at_pit': tire_age_at_pit,
                    'compound_num': compound_num,
                    'race_progress': round(race_progress, 3),
                    'position_before': pos_before,
                    'laps_remaining': total_laps - pit_lap,
                    'position_delta': position_delta,
                    'optimal_pit': optimal
                })

    return pd.DataFrame(records)

print("Building optimal pit dataset...")
optimal_df = build_optimal_pit_dataset(df_recent)
print(f"Total pit stops analyzed: {optimal_df.shape[0]}")
print(f"Optimal pit rate: {optimal_df['optimal_pit'].mean():.1%}")
print(optimal_df.head(10))

Building optimal pit dataset...
Total pit stops analyzed: 941
Optimal pit rate: 47.2%
          race driver  pit_lap  tire_age_at_pit  compound_num  race_progress  \
0  Bahrain2022    VER     15.0             17.0             0          0.263   
1  Bahrain2022    VER     31.0             16.0             0          0.544   
2  Bahrain2022    VER     44.0             13.0             1          0.772   
3  Bahrain2022    GAS     15.0             17.0             0          0.263   
4  Bahrain2022    GAS     33.0             18.0             1          0.579   
5  Bahrain2022    PER     16.0             18.0             0          0.281   
6  Bahrain2022    PER     34.0             18.0             1          0.596   
7  Bahrain2022    PER     44.0             13.0             0          0.772   
8  Bahrain2022    ALO     12.0             14.0             0          0.211   
9  Bahrain2022    ALO     26.0             14.0             1          0.456   

   position_before  laps_remainin

In [14]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score

FEATURES = [
    'tire_age_at_pit', 'compound_num',
    'race_progress', 'position_before', 'laps_remaining'
]
TARGET = 'optimal_pit'

X = optimal_df[FEATURES]
y = optimal_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model_gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
model_gb.fit(X_train, y_train)

model_rf = RandomForestClassifier(
    n_estimators=300, max_depth=6,
    random_state=42
)
model_rf.fit(X_train, y_train)

for name, model in [('GradientBoosting', model_gb), ('RandomForest', model_rf)]:
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n{name}:")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")

fi = pd.Series(model_gb.feature_importances_, index=FEATURES).sort_values(ascending=False)
print(f"\nFeature importances (GradientBoosting):")
print(fi.round(3))


GradientBoosting:
              precision    recall  f1-score   support

           0       0.71      0.72      0.72       100
           1       0.68      0.67      0.68        89

    accuracy                           0.70       189
   macro avg       0.70      0.70      0.70       189
weighted avg       0.70      0.70      0.70       189

ROC-AUC: 0.745

RandomForest:
              precision    recall  f1-score   support

           0       0.62      0.80      0.70       100
           1       0.67      0.46      0.55        89

    accuracy                           0.64       189
   macro avg       0.65      0.63      0.62       189
weighted avg       0.65      0.64      0.63       189

ROC-AUC: 0.701

Feature importances (GradientBoosting):
race_progress      0.350
position_before    0.246
tire_age_at_pit    0.186
laps_remaining     0.175
compound_num       0.042
dtype: float64


In [15]:
import joblib, pandas as pd

# Save strategy model
joblib.dump(model_gb, 'models/strategy_model.pkl')

# Save optimal pit dataframe
optimal_df.to_csv('data/optimal_pit_df.csv', index=False)

print("Saved!")
print(f"  models/strategy_model.pkl")
print(f"  data/optimal_pit_df.csv  ({optimal_df.shape[0]} rows)")

Saved!
  models/strategy_model.pkl
  data/optimal_pit_df.csv  (941 rows)
